# Traffic Sign Detection - YOLOv8 (Real-World Ready)
Better for camera & internet images

In [ ]:
!pip install -q ultralytics

## 1. Download & Prepare GTSRB for YOLO

In [ ]:
from google.colab import files
import zipfile, os
from pathlib import Path

print('Upload GTSRB zip')
uploaded = files.upload()

for f in uploaded:
    if f.endswith('.zip'):
        with zipfile.ZipFile(f) as z:
            z.extractall('/content/')

!mv /content/Train /content/gtsrb_raw 2>/dev/null || true
print('Extracted')

In [ ]:
import cv2
from tqdm import tqdm

src = '/content/gtsrb_raw'
dst = '/content/yolo_dataset'
os.makedirs(f'{dst}/images/train', exist_ok=True)
os.makedirs(f'{dst}/labels/train', exist_ok=True)

classes = sorted([d for d in os.listdir(src) if d.isdigit()], key=int)

for cls_id, cls in enumerate(tqdm(classes)):
    for img_name in os.listdir(os.path.join(src, cls)):
        if not img_name.lower().endswith(('.png','.jpg','.ppm','.jpeg')):
            continue
        img_path = os.path.join(src, cls, img_name)
        img = cv2.imread(img_path)
        if img is None: continue
        label = f"{cls_id} 0.5 0.5 1.0 1.0\n"
        out_img = f'{dst}/images/train/{cls}_{img_name}'
        out_lbl = f'{dst}/labels/train/{cls}_{Path(img_name).stem}.txt'
        cv2.imwrite(out_img, img)
        with open(out_lbl, 'w') as f:
            f.write(label)

print('YOLO dataset ready')

In [ ]:
yaml = '''
path: /content/yolo_dataset
train: images/train
val: images/train

nc: 43
names: ['20','30','50','60','70','80','end80','100','120','no_pass','no_pass_3.5','right_of_way','priority','yield','stop','no_veh','no_3.5','no_entry','caution','curve_left','curve_right','double_curve','bumpy','slippery','narrow','work','signals','pedestrian','children','bicycle','ice','animals','end_limits','turn_right','turn_left','ahead','straight_right','straight_left','keep_right','keep_left','roundabout','end_no_pass','end_no_pass_3.5']
'''
with open('/content/data.yaml','w') as f:
    f.write(yaml)
print('yaml created')

## 2. Train YOLOv8

In [ ]:
from ultralytics import YOLO

model = YOLO('yolov8n.pt')
model.train(
    data='/content/data.yaml',
    epochs=20,
    imgsz=224,
    batch=32,
    name='gtsrb_yolo',
    patience=5
)
print('Training done')

## 3. Download Model

In [ ]:
from google.colab import files
files.download('/content/runs/detect/gtsrb_yolo/weights/best.pt')